# 01 — Build PrimeKG-Plus from source databases

This notebook assembles PrimeKG-Plus from updated upstream dumps (build tag **`RUN_DATE`**).

**Pipeline**
1. Load and standardize edge tables (PPI, DrugBank, DisGeNET, Open Targets, DrugCentral, MONDO/HPO, Bgee, GO, CTD, SIDER/nSIDES, literature, etc.).
2. Concatenate into a unified edge list (`kg`), add reverse edges, deduplicate, and remove self-loops.
3. Export `auxillary/{RUN_DATE}-kg_raw.csv`, extract the **giant connected component**, and write `auxillary/{RUN_DATE}-kg_giant.csv`.

**Next step:** run `02_disease_grouping.ipynb` on `kg_giant.csv`.

Configure paths in the **first code cell** below. The notebook auto-detects `PROJECT_ROOT` from the current working directory (works when opened from `PrimeKG-Plus_release/scripts/` or `knowledge_graph/`). Override with `export PRIMEKG_ROOT=/path/to/PrimeKG` if needed.


In [77]:
import os
from pathlib import Path

import igraph as ig
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm


def resolve_primekg_root() -> Path:
    """Locate PrimeKG repo root (contains datasets/data), from knowledge_graph/ or PrimeKG-Plus_release/scripts/."""
    if env := os.environ.get("PRIMEKG_ROOT"):
        root = Path(env).expanduser().resolve()
        if (root / "datasets" / "data" / "ppi").is_dir():
            return root
        raise FileNotFoundError(f"PRIMEKG_ROOT invalid (no datasets/data): {root}")

    for start in (Path.cwd().resolve(),):
        for parent in (start, *start.parents):
            if (parent / "datasets" / "data" / "ppi").is_dir():
                return parent
            if parent.name == "PrimeKG-Plus_release" and (parent.parent / "datasets" / "data" / "ppi").is_dir():
                return parent.parent

    raise FileNotFoundError(
        "Cannot find PrimeKG source data. Export PRIMEKG_ROOT to the repository root "
        "(the folder that contains datasets/data/)."
    )


PROJECT_ROOT = resolve_primekg_root()
DATA_DIR = PROJECT_ROOT / "datasets" / "data"
KG_DIR = DATA_DIR / "kg"
KG_SCRIPTS_DIR = PROJECT_ROOT / "knowledge_graph"
RELEASE_ROOT = PROJECT_ROOT / "PrimeKG-Plus_release"
RP_INDICATION_CSV = RELEASE_ROOT / "source_prep/repurposed_drug/outputs/20260405-RepurposedDrug_Indication.csv"
SIDER_WITH_NSIDES_CSV = RELEASE_ROOT / "source_prep/sider_nsides/outputs/sider_with_nsides.csv"

# String paths kept for compatibility with existing read_csv calls in this notebook
data_path = str(DATA_DIR) + os.sep
save_path = str(KG_DIR) + os.sep
RUN_DATE = "20260529"

_ppi = DATA_DIR / "ppi" / "20260426-protein_protein.csv"
if not _ppi.is_file():
    raise FileNotFoundError(f"Missing PPI source file: {_ppi}")


# Read datasets

## Phase 1 — Load auxiliary edge sources (PPI, DrugBank drug–protein)

These are smaller “side” tables loaded early: **protein–protein interactions** and **DrugBank drug–protein** relations. They feed the **Basic** edge section later (`df_ppi`, `df_drugbank`).


In [19]:
def assert_dtypes(df):
    """Raise if any column is not string/object dtype."""
    bad = []
    for i, x in enumerate(df.dtypes.values):
        if x != np.dtype("O"):
            bad.append(f"{df.columns[i]}: {x}")
    if bad:
        raise AssertionError("Non-string columns: " + ", ".join(bad))

In [20]:
df_ppi = pd.read_csv(data_path+'ppi/20260426-protein_protein.csv', low_memory=False).dropna()
df_ppi = df_ppi.astype({'proteinA_entrezid':int}).astype({'proteinA_entrezid':str})
df_ppi = df_ppi.astype({'proteinB_entrezid':int}).astype({'proteinB_entrezid':str})
assert_dtypes(df_ppi)

In [21]:
df_drugbank = pd.read_csv(data_path+'/drugbank/Nurset_data_drugbank/drug_protein.csv', low_memory=False)
df_drugbank = df_drugbank.get(['DrugBank', 'relation', 'NCBIGeneID','DrugBankName']).dropna()
df_drugbank = df_drugbank.astype({'NCBIGeneID':int}).astype({'NCBIGeneID':str})
assert_dtypes(df_drugbank)

## DisGeNET + OpenTargets gene–disease / phenotype tables

- DisGeNET: `disgenet/Authors-curated_gene_disease_associations.tsv`
- OpenTargets: `disgenet/OpenTarget/20260426-OpenTarget_disease_protein_associations.csv`
  - **How this file was built:** `PrimeKG-Plus_release/source_prep/opentarget/process_opentarget.ipynb` (see `source_prep/opentarget/README.md`)
- Concatenate both sources, then `drop_duplicates(subset=["diseaseName", "geneSymbol"])` before building protein–disease / protein–phenotype edges.


In [22]:
df_disgenet = pd.read_csv(data_path+'disgenet/Authors-curated_gene_disease_associations.tsv', sep='\t', low_memory=False)
df_disgenet = df_disgenet.astype({'geneId':int}).astype({'geneId':str})

In [24]:
df_disgenet.diseaseType.unique()

array(['phenotype', 'disease', 'group'], dtype=object)

In [25]:
df_disgenet = df_disgenet[['geneId', 'score', 'diseaseName', 'diseaseId', 'geneSymbol', 'diseaseType']]

In [26]:
opentarget_df = pd.read_csv(str(DATA_DIR / 'disgenet/OpenTarget/20260426-OpenTarget_disease_protein_associations.csv'))
opentarget_df["diseaseType"] = "disease"
opentarget_df = opentarget_df[['x_id', 'score', 'diseaseName', 'diseaseId', 'x_name',  'diseaseType']]
opentarget_df.columns = ['geneId', 'score', 'diseaseName', 'diseaseId', 'geneSymbol', 'diseaseType']
opentarget_df = opentarget_df.dropna(subset=["geneId"])
opentarget_df["geneId"] = opentarget_df["geneId"].astype(int)

In [27]:
df_disgenet_opentarget = pd.concat([df_disgenet, opentarget_df], axis=0)

In [28]:
df_disgenet_opentarget = df_disgenet_opentarget.drop_duplicates(subset=["diseaseName", "geneSymbol"])
len(df_disgenet_opentarget)

353757

## MONDO disease labels (`mondo_terms`)

Loads MONDO term names/ids used as the **canonical disease label** table after mapping sources to `mondo_id`.

Includes lightweight **QC** (`id` duplicates, obsolete flags if present) and optional name-based sanity checks.


In [31]:
df_mondo_terms = pd.read_csv(data_path+'mondo/20251124-mondo_terms.csv', low_memory=False)
df_mondo_terms = df_mondo_terms.astype({'id':str})
a=df_mondo_terms.name.unique()

In [32]:
df_mondo_terms['id'].duplicated().any()

False

In [34]:
df_mondo_terms['name'].duplicated().any()

False

## MONDO cross-references (`mondo_references`)

- **`ontology == UMLS` → `df_mondo_xref_umls`:** bridge **CUI ↔ MONDO** for disease normalization.
- **`ontology == HP` → `df_mondo_xref_hp`:** HP phenotypes that MONDO also treats in a disease context (used to **rewrite** phenotype-heavy edges to disease nodes via `mondo_xref_hp_subset`).

The duplicate-QC cells detect HP↔MONDO patterns:
- **`ontology_id` duplicated (1 HP → many MONDO):** must be resolved before graph build (drop incorrect MONDO rows, e.g. `9813`, `18767`).
- **`mondo_id` duplicated (1 MONDO → many HP):** expected; each HP still maps to one MONDO, so downstream merges on `ontology_id` are not inflated.


In [35]:
df_mondo_xref = pd.read_csv(data_path+'mondo/20251124-mondo_references.csv', low_memory=False)
df_mondo_xref = df_mondo_xref.astype({'mondo_id':int}).astype({'mondo_id':str})
assert_dtypes(df_mondo_xref)

In [36]:
df_mondo_xref_umls =  df_mondo_xref[df_mondo_xref.ontology=="UMLS"]

In [37]:
df_mondo_xref_hp = df_mondo_xref[df_mondo_xref.ontology=="HP"]

In [38]:
df_mondo_xref_hp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 579 entries, 65 to 130442
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ontology_id  579 non-null    object
 1   ontology     579 non-null    object
 2   mondo_id     579 non-null    object
dtypes: object(3)
memory usage: 18.1+ KB


In [39]:
df_mondo_xref_hp.ontology_id.duplicated().any()

True

In [40]:
df_mondo_xref_hp.ontology_id.duplicated().sum()

2

In [41]:
df_mondo_xref_hp.mondo_id.duplicated().sum()

5

In [42]:
# Duplicates by ontology_id
dup_ontology = df_mondo_xref_hp[df_mondo_xref_hp.ontology_id.duplicated(keep=False)]

# Duplicates by mondo_id
dup_mondo = df_mondo_xref_hp[df_mondo_xref_hp.mondo_id.duplicated(keep=False)]

# Duplicates by both columns (exact duplicate)
dup_both = df_mondo_xref_hp[df_mondo_xref_hp.duplicated(subset=["ontology_id", "mondo_id"], keep=False)]

In [43]:
df_mondo_xref_hp = df_mondo_xref_hp.loc[~df_mondo_xref_hp["mondo_id"].isin(["9813", "18767"])] #"9813", "18767" are not correct

In [44]:
mondo_multi = (
    df_mondo_xref_umls.groupby("mondo_id")["ontology_id"]
    .agg(list)
    .reset_index()
    .rename(columns={"ontology_id": "ontology_ids"})
)
mondo_multi = mondo_multi[mondo_multi["ontology_ids"].apply(len) > 1]


In [45]:
df_mondo_xref_umls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21381 entries, 6 to 136796
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ontology_id  21381 non-null  object
 1   ontology     21381 non-null  object
 2   mondo_id     21381 non-null  object
dtypes: object(3)
memory usage: 668.2+ KB


In [46]:
df_mondo_xref_umls[df_mondo_xref_umls.mondo_id.isin(["13509", "1999"])]

,ontology_id,ontology,mondo_id
9076,C0152171,UMLS,1999
9078,C3203102,UMLS,1999
78381,C3151411,UMLS,13509
78382,C5681638,UMLS,13509


In [47]:
#remove invalid pairs
df_mondo_xref_umls = df_mondo_xref_umls[~df_mondo_xref_umls.ontology_id.isin(["C0152171", "C5681638"])]

In [49]:
df_mondo_parents = pd.read_csv(data_path+'mondo/20251124-mondo_parents.csv', low_memory=False)
df_mondo_parents = df_mondo_parents.astype({'parent':str})
df_mondo_parents = df_mondo_parents.astype({'child':str})
assert_dtypes(df_mondo_parents)

In [50]:
df_mondo_xref_hp.ontology_id.duplicated().any()

False

In [51]:
df_mondo_xref_hp.ontology_id.duplicated().sum()

0

In [52]:
df_mondo_xref_hp.mondo_id.duplicated().sum()

5

## DrugCentral + repurposed indications (DrugBank IDs)

Loads cleaned DrugCentral drug–disease rows and (optionally) merges **repurposed indication** rows, with an **anti-join** to avoid duplicating edges already present in DrugCentral.


In [73]:
df_drug_central = pd.read_csv(data_path+'drugcentral/05Oct2023_drug_disease_cleaned.csv')

In [74]:
df_drug_central[df_drug_central.name=="ibrexafungerp"]

,cd_id,cd_formula,cd_molweight,id,clogp,alogs,cas_reg_no,tpsa,lipinski,name,...,status,id.1,struct_id,concept_id,relationship_name,concept_name,umls_cui,snomed_full_name,cui_semantic_type,snomed_conceptid
41811,5768,C44H67N5O4,730.051,5462,6.03,-6.32,1207753-03-4,125.38,2.0,ibrexafungerp,...,ONP,174546,5462,40249220,indication,Candidiasis of vagina,C0006852,Candidiasis of vagina,T047,72934000.0
41812,5768,C44H67N5O4,730.051,5462,6.03,-6.32,1207753-03-4,125.38,2.0,ibrexafungerp,...,ONP,174547,5462,40249487,indication,Vulvovaginal candidiasis,C0700345,Candidal vulvovaginitis,T047,72605008.0
41813,5768,C44H67N5O4,730.051,5462,6.03,-6.32,1207753-03-4,125.38,2.0,ibrexafungerp,...,ONP,174548,5462,21003446,contraindication,"Pregnancy, function",C0032961,"Pregnancy, function",T040,289908002.0


In [75]:
# df_drug_central = pd.read_csv(data_path+'drugcentral/drug_disease_latest.csv', low_memory=False)
df_drug_central = pd.read_csv(data_path+'drugcentral/05Oct2023_drug_disease_cleaned.csv') # 'concept_id', 'concept_name', 'snomed_conceptid'
df_drug_central = df_drug_central.query('not @df_drug_central.cas_reg_no.isna()')
df_drug_central = df_drug_central.query('not @df_drug_central.umls_cui.isna()')
df_drug_central = df_drug_central[["cas_reg_no", "umls_cui", "relationship_name"]]
assert_dtypes(df_drug_central)
df_drug_central = df_drug_central.drop_duplicates()

In [78]:
rp_indication = pd.read_csv(RP_INDICATION_CSV)
rp_indication


,cas_reg_no,umls_cui,relationship_name
0,616-91-1,C0022660,indication
1,55079-83-9,C0033860,indication
2,142340-99-6,C0019163,indication
3,142340-99-6,C0524909,indication
4,106941-25-7,C0019163,indication
...,...,...,...
443,118072-93-8,C0029458,indication
444,139264-17-8,C0338480,indication
445,82626-48-0,C0021603,indication
446,68291-97-4,C0014544,indication


In [79]:
# Anti-join repurposed edges against DrugCentral by ID triple to avoid duplicate KG edges.
KEYS = ["cas_reg_no", "umls_cui", "relationship_name"]

new_rp_indication = rp_indication.merge(
    df_drug_central[KEYS].assign(_in_dc=1),
    on=KEYS,
    how="left",
)
new_rp_indication = new_rp_indication[new_rp_indication["_in_dc"].isna()].drop(columns=["_in_dc"])

df_drug_central = pd.concat([df_drug_central, new_rp_indication], ignore_index=True)


In [80]:
pwd

'/Users/ljw303/YANG_DATA/PrimeKG/PrimeKG-Plus_release/scripts'

In [81]:
# WILL BE INCLUDED in a section about information retrieved from DrugBank
df_ddi = pd.read_csv(data_path+'drugbank/Nurset_data_drugbank/drug_drug.csv', low_memory=False)
assert_dtypes(df_ddi)

## HPO resources (terms, structure, disease annotations)

Loads HP term metadata, HP cross-refs, parent edges, and **HPOA disease–phenotype positives** used to build disease–phenotype edges before MONDO-aware rewiring.


In [82]:
df_hp_terms = pd.read_csv(data_path+'hpo/20251124-hp_terms.csv', low_memory=False)
df_hp_terms = df_hp_terms.astype({'id':int}).astype({'id':str})

In [85]:
df_hp_xref = pd.read_csv(data_path+'hpo/20251124-hp_references.csv', low_memory=False)
df_hp_xref = df_hp_xref.astype({'hp_id':int}).astype({'hp_id':str})

In [86]:
df_hp_parents = pd.read_csv(data_path+'hpo/20251124-hp_parents.csv', low_memory=False)
df_hp_parents = df_hp_parents.astype({'parent':int}).astype({'parent':str})
df_hp_parents = df_hp_parents.astype({'child':int}).astype({'child':str})
assert_dtypes(df_hp_parents)

In [87]:
df_hpoa_pos = pd.read_csv(data_path+'hpo/20251124-disease_phenotype_pos.csv', low_memory=False)
df_hpoa_pos = df_hpoa_pos.astype({'hp_id':int}).astype({'hp_id':str})
df_hpoa_pos = df_hpoa_pos.astype({'disease_ontology_id':int}).astype({'disease_ontology_id':str})
assert_dtypes(df_hpoa_pos)

df_hpoa_neg = pd.read_csv(data_path+'hpo/20251124-disease_phenotype_neg.csv', low_memory=False)
df_hpoa_neg = df_hpoa_neg.astype({'hp_id':int}).astype({'hp_id':str})
df_hpoa_neg = df_hpoa_neg.astype({'disease_ontology_id':int}).astype({'disease_ontology_id':str})
assert_dtypes(df_hpoa_neg)

## SIDER drug–phenotype (effect) edges

Loads SIDER-style tables used later for **drug ↔ phenotype / effect** relations.


In [ ]:
df_sider = pd.read_csv(SIDER_WITH_NSIDES_CSV, low_memory=False)
assert_dtypes(df_sider)


## Gene Ontology (GO)

Loads three related tables: **`df_go_terms`** (GO IDs, names, and BP/MF/CC typing), **`df_go_edges`** (GO-term–GO-term relations), and **`df_gene2go`** (NCBI gene ↔ GO term associations from the `ncbigene` bundle). Later Phase-2 cells expand BP/MF/CC subgraphs and merge gene–GO (and CTD-derived GO) rows into **`clean_edges`**-style edge frames.


In [ ]:
df_go_terms = pd.read_csv(data_path+'go/20251124-go_terms_info.csv', low_memory=False)
df_go_terms = df_go_terms.astype({'go_term_id':int}).astype({'go_term_id':str})
assert_dtypes(df_go_terms)

In [ ]:
df_go_edges = pd.read_csv(data_path+'go/20251124-go_terms_relations.csv', low_memory=False)
df_go_edges = df_go_edges.astype({'x':int}).astype({'x':str})
df_go_edges = df_go_edges.astype({'y':int}).astype({'y':str})
assert_dtypes(df_go_edges)

In [ ]:
df_gene2go = pd.read_csv(data_path+'ncbigene/20251124-protein_go_associations.csv', low_memory=False)
df_gene2go = df_gene2go.astype({'ncbi_gene_id':int}).astype({'ncbi_gene_id':str})
df_gene2go = df_gene2go.astype({'go_term_id':int}).astype({'go_term_id':str})
assert_dtypes(df_gene2go)

## CTD (Comparative Toxicogenomics Database)

Loads **`df_exposures`** from the CTD export (chemical / exposure–gene–disease–phenotype–pathway associations, depending on columns). Later steps filter and reshape this into **exposure/chemical ↔ gene or disease** style edges aligned with the graph’s node ID conventions.


In [ ]:
df_exposures = pd.read_csv(data_path+'ctd/20251124-exposure_data.csv', low_memory=False)
df_exposures = df_exposures.get(['exposurestressorname', 'exposurestressorid',
                  'exposuremarker', 'exposuremarkerid',
                  'diseasename', 'diseaseid',
                  'phenotypename', 'phenotypeid'])
assert_dtypes(df_exposures)

## UBERON (anatomy ontology)

Loads **`df_uberon_terms`** (anatomy term labels) plus **`df_uberon_is_a`** (hierarchy) and **`df_uberon_rels`** (other relations). These IDs are the bridge for **anatomy ↔ expression (BGEE)** and anatomy-aware disease / phenotype normalization elsewhere in the notebook.


In [ ]:
df_uberon_terms = pd.read_csv(data_path+'uberon/20251124-uberon_terms.csv', low_memory=False)
df_uberon_terms = df_uberon_terms.astype({'id':int}).astype({'id':str})
assert_dtypes(df_uberon_terms)

In [ ]:
df_uberon_is_a = pd.read_csv(data_path+'uberon/20251124-uberon_is_a.csv', low_memory=False)
df_uberon_is_a = df_uberon_is_a.astype({'id':int}).astype({'id':str})
df_uberon_is_a = df_uberon_is_a.astype({'is_a':int}).astype({'is_a':str})
assert_dtypes(df_uberon_is_a)

In [ ]:
df_uberon_rels = pd.read_csv(data_path+'uberon/20251124-uberon_rels.csv', low_memory=False)
df_uberon_rels = df_uberon_rels.astype({'id':int}).astype({'id':str})
df_uberon_rels = df_uberon_rels.astype({'relation_id':int}).astype({'relation_id':str})
assert_dtypes(df_uberon_rels)

## BGEE anatomy–gene expression

Reads **`anatomy_gene.csv`** and applies an **expression rank** cutoff to limit edge volume before building anatomy–protein edges.


In [ ]:
df_bgee = pd.read_csv(data_path+'bgee/anatomy_gene.csv', low_memory=False)
df_bgee = df_bgee.astype({'expression_rank':int}).astype({'expression_rank':str})
df_bgee = df_bgee.astype({'anatomy_id':int}).astype({'anatomy_id':str})
assert_dtypes(df_bgee)
df_bgee = df_bgee.drop_duplicates()

In [ ]:
expression_rank = 5000 # if choose 10000: 2038870 rows, if choose 12000: 2446042 rows, 
#if choose 25000: 5249452 rows, if choose 15000: 3059314 rows
df_bgee = df_bgee[pd.to_numeric(df_bgee["expression_rank"], errors="coerce") <= expression_rank]
len(df_bgee)

## Reactome (pathways)

Loads **`df_reactome_terms`** (pathway labels/metadata), **`df_reactome_rels`** (pathway–pathway graph edges), and **`df_reactome_ncbi`** (Reactome stable IDs ↔ NCBI Gene). Downstream merges attach **genes/proteins ↔ pathways** and pathway names into the standardized edge tables.


In [ ]:
df_reactome_terms = pd.read_csv(data_path+'reactome/20251124-reactome_terms.csv', low_memory=False)
assert_dtypes(df_reactome_terms)

In [ ]:
df_reactome_rels = pd.read_csv(data_path+'reactome/20251124-reactome_relations.csv', low_memory=False)
assert_dtypes(df_reactome_rels)

In [ ]:
df_reactome_ncbi = pd.read_csv(data_path+'reactome/20251124-reactome_ncbi.csv', low_memory=False)
df_reactome_ncbi['ncbi_id'] = df_reactome_ncbi['ncbi_id'].astype(str)
df_reactome_ncbi = df_reactome_ncbi[df_reactome_ncbi.ncbi_id.str.isnumeric()]
assert_dtypes(df_reactome_ncbi)

## UMLS–MONDO vocabulary (`df_umls_mondo`)

Loads the curated **`(umls_id, mondo_id)`** table used wherever a source encodes disease as a **UMLS CUI** and the graph standardizes on **MONDO**.

- **Release build:** `20260510-umls_mondo_no_multi_mapping_v2.csv` (**21,357** bijective pairs; Monarch + UMLS TS preferred CUI per MONDO).
- Earlier no-multi-mapping v1 (`20251124-umls_mondo_no_multi_mapping.csv`, 21,347 pairs) is kept in `datasets/data/vocab/` for comparison only.
- Typical merges use **`inner`** so only rows with a resolvable CUI→MONDO survive.


In [ ]:
# df_umls_mondo = pd.read_csv(data_path+'vocab/20251124-umls_mondo.csv', low_memory=False)  # original PrimeKG (multi-mapping)
# df_umls_mondo = pd.read_csv(data_path+'vocab/20251124-umls_mondo_no_multi_mapping.csv', low_memory=False)  # v1 (21,347 pairs)
df_umls_mondo = pd.read_csv(data_path+'vocab/20260510-umls_mondo_no_multi_mapping_v2.csv', low_memory=False)
df_umls_mondo = df_umls_mondo.astype({'mondo_id':int}).astype({'mondo_id':str})
assert len(df_umls_mondo) == df_umls_mondo['umls_id'].nunique() == df_umls_mondo['mondo_id'].nunique()
assert_dtypes(df_umls_mondo)


In [ ]:
df_umls_mondo.info()

In [ ]:
df_prot_names = pd.read_csv(data_path+'vocab/20251124-gene_names.csv', low_memory=False, sep='\t')
df_prot_names = df_prot_names.rename(columns={'NCBI Gene ID(supplied by NCBI)':'ncbi_id', 'NCBI Gene ID':'ncbi_id2', 'Approved symbol':'symbol', 'Approved name':'name'})
df_prot_names = df_prot_names.get(['ncbi_id', 'symbol']).dropna()
df_prot_names = df_prot_names.astype({'ncbi_id':int}).astype({'ncbi_id':str})
assert_dtypes(df_prot_names)

In [ ]:
db_vocab = pd.read_csv(data_path+'vocab/20251124-drugbank_vocabulary.csv', low_memory=False)
assert_dtypes(db_vocab)

In [ ]:
df_db_atc = pd.read_csv(data_path+'vocab/20251124-drugbank_atc_codes.csv', low_memory=False).get(['atc_code','parent_key'])
assert_dtypes(df_db_atc)

## Phase 2 — Build standardized edge tables

Each subsection below turns one **resource family** into a small set of columns (`relation`, `display_relation`, `x_*`, `y_*`) and usually ends with **`clean_edges()`** (drop nulls, dedupe, remove trivial self-loops).

**Order matters:** disease normalization (`df_umls_mondo`, `df_mondo_terms`) must exist **before** merges that attach MONDO names to drug–disease or protein–disease edges; HP→MONDO rewiring must run **after** phenotype tables are assembled.


# Converting databases into graph edges

In [ ]:
# Function to clean and standardize edge dataframes
# This function standardizes edge dataframes by:
# 1. Selecting only required columns (relation, display_relation, x/y metadata)
# 2. Removing rows with missing values
# 3. Removing duplicate edges
# 4. Removing self-loops (edges where x and y are identical)

def clean_edges(df): 
    df = df.get(['relation', 'display_relation', 'x_id','x_type', 'x_name', 'x_source','y_id','y_type', 'y_name', 'y_source'])
    df = df.dropna()
    df = df.drop_duplicates()
    df = df.query('not ((x_id == y_id) and (x_type == y_type) and (x_source == y_source) and (x_name == y_name))')
    return df

## Basic

### Protein protein interactions (NCBI)

In [ ]:
# Create protein-protein interaction edges
# Merges PPI data with gene names to create edges between proteins.
# Sets metadata: relation='protein_protein', display_relation='ppi', source='NCBI', type='gene/protein'
# Both x and y nodes represent proteins from NCBI Gene database.

df_prot_prot = pd.merge(df_ppi, df_prot_names, 'left', left_on='proteinA_entrezid', right_on='ncbi_id').rename(columns={'symbol':'symbolA'})
df_prot_prot = pd.merge(df_prot_prot, df_prot_names, 'left', left_on='proteinB_entrezid', right_on='ncbi_id').rename(columns={'symbol':'symbolB'})
df_prot_prot = df_prot_prot.rename(columns={'proteinA_entrezid':'x_id', 'proteinB_entrezid':'y_id', 'symbolA':'x_name', 'symbolB':'y_name'})
df_prot_prot['x_type'] = 'gene/protein'
df_prot_prot['x_source'] = 'NCBI'
df_prot_prot['y_type'] = 'gene/protein'
df_prot_prot['y_source'] = 'NCBI'
df_prot_prot['relation'] = 'protein_protein'
df_prot_prot['display_relation'] = 'ppi'
df_prot_prot = clean_edges(df_prot_prot)

### Drug protein interactions (DrugBank)

In [ ]:
# Create drug-protein interaction edges
# Merges DrugBank drug-protein associations with gene names.
# Creates edges from drugs (DrugBank) to proteins (NCBI).
# Combines all drug-protein relation types (target, carrier, enzyme, transporter) into 'drug_protein' relation.
df_prot_drug = pd.merge(df_drugbank, df_prot_names, 'left', left_on='NCBIGeneID', right_on='ncbi_id')
df_prot_drug = df_prot_drug.rename(columns={'DrugBank':'x_id', 'NCBIGeneID':'y_id', 'DrugBankName':'x_name', 'symbol':'y_name'})
df_prot_drug['x_type'] = 'drug'
df_prot_drug['x_source'] = 'DrugBank'
df_prot_drug['y_type'] = 'gene/protein'
df_prot_drug['y_source'] = 'NCBI'
df_prot_drug['display_relation'] = df_prot_drug.get('relation').values
df_prot_drug['relation'] = 'drug_protein' # combine targets, carrier, enzyme and transporter
df_prot_drug = clean_edges(df_prot_drug)

### Drug disease interactions (DrugCentral)

In [ ]:
df_drug_central[df_drug_central.cas_reg_no=="10102-43-9"]

In [ ]:
df_drug_dis = pd.merge(df_drug_central, db_vocab, 'left', left_on='cas_reg_no', right_on='CAS')

In [ ]:
df_drug_dis[df_drug_dis.cas_reg_no=="10102-43-9"]

In [ ]:
df_drug_dis = pd.merge(df_drug_dis, df_umls_mondo, 'inner', left_on='umls_cui', right_on='umls_id')

In [ ]:
df_drug_dis = pd.merge(df_drug_dis, df_mondo_terms, 'left', left_on='mondo_id', right_on='id')

In [ ]:
df_drug_dis = df_drug_dis.get(['relationship_name', 'DrugBank ID', 'Common name', 'mondo_id', 'name'])
df_drug_dis = df_drug_dis.dropna().drop_duplicates()

df_drug_dis = df_drug_dis.rename(columns={'DrugBank ID':'x_id', 'mondo_id':'y_id', 'Common name':'x_name', 'name':'y_name', 'relationship_name':'relation'})
df_drug_dis['x_type'] = 'drug'
df_drug_dis['x_source'] = 'DrugBank'
df_drug_dis['y_type'] = 'disease'
df_drug_dis['y_source'] = 'MONDO'
df_drug_dis['display_relation'] = df_drug_dis.get('relation').values
df_drug_dis = clean_edges(df_drug_dis)

### Disease protein interactions (DisGenNet)

In [ ]:
df_prot_dis1 = df_disgenet_opentarget.query('diseaseType=="disease"')
df_prot_dis1 = pd.merge(df_prot_dis1, df_umls_mondo, 'inner', left_on='diseaseId', right_on='umls_id')
df_prot_dis1 = pd.merge(df_prot_dis1, df_mondo_terms, 'left', left_on='mondo_id', right_on='id')

In [ ]:
df_prot_dis1 = df_prot_dis1.rename(columns={'geneId':'x_id', 'geneSymbol':'x_name', 'mondo_id':'y_id', 'name':'y_name'})
df_prot_dis1['x_type'] = 'gene/protein'
df_prot_dis1['x_source'] = 'NCBI'
df_prot_dis1['y_type'] = 'disease'
df_prot_dis1['y_source'] = 'MONDO'
df_prot_dis1['relation'] = 'disease_protein'
df_prot_dis1['display_relation'] = 'associated with'
df_prot_dis1 = clean_edges(df_prot_dis1)

### Disease disease interations (MONDO)

In [ ]:
df_dis_dis1 = pd.merge(df_mondo_parents, df_mondo_terms, 'left', left_on='parent', right_on='id')
df_dis_dis1 = df_dis_dis1.rename(columns={'parent':'x_id', 'name':'x_name'})
df_dis_dis1 = pd.merge(df_dis_dis1, df_mondo_terms, 'left', left_on='child', right_on='id')
df_dis_dis1 = df_dis_dis1.rename(columns={'child':'y_id', 'name':'y_name'})
df_dis_dis1['x_type'] = 'disease'
df_dis_dis1['x_source'] = 'MONDO'
df_dis_dis1['y_type'] = 'disease'
df_dis_dis1['y_source'] = 'MONDO'
df_dis_dis1['relation'] = 'disease_disease'
df_dis_dis1['display_relation'] = 'parent-child'
df_dis_dis1 = clean_edges(df_dis_dis1)

### Drug drug interactions (DrugBank)

In [ ]:
df_drug_drug = pd.merge(df_ddi, db_vocab, 'inner', left_on='drug1', right_on='DrugBank ID')
df_drug_drug = df_drug_drug.rename(columns={'drug1':'x_id', 'Common name':'x_name'})
df_drug_drug = pd.merge(df_drug_drug.astype({'drug2':'str'}), db_vocab, 'inner', left_on='drug2', right_on='DrugBank ID')
df_drug_drug = df_drug_drug.rename(columns={'drug2':'y_id', 'Common name':'y_name'})
df_drug_drug['x_type'] = 'drug'
df_drug_drug['x_source'] = 'DrugBank'
df_drug_drug['y_type'] = 'drug'
df_drug_drug['y_source'] = 'DrugBank'
df_drug_drug['relation'] = 'drug_drug'
df_drug_drug['display_relation'] = 'synergistic interaction'
df_drug_drug = clean_edges(df_drug_drug)

## Effect/Phenotype

### Effect protein interactions (DisGenNet)

In [ ]:
df_hp_xref.ontology.unique()

In [ ]:
df_hp_terms.name.unique()

In [ ]:
df_prot_phe = df_disgenet_opentarget.query('diseaseType=="phenotype"')
df_prot_phe = pd.merge(df_prot_phe, df_hp_xref, 'inner', left_on='diseaseId', right_on='ontology_id')
df_prot_phe = pd.merge(df_prot_phe, df_hp_terms, 'left', left_on='hp_id', right_on='id')

df_prot_phe = df_prot_phe.rename(columns={'geneId':'x_id', 'geneSymbol':'x_name', 'hp_id':'y_id', 'name':'y_name'})
df_prot_phe['x_type'] = 'gene/protein'
df_prot_phe['x_source'] = 'NCBI'
df_prot_phe['y_type'] = 'effect/phenotype'
df_prot_phe['y_source'] = 'HPO'
df_prot_phe['relation'] = 'phenotype_protein'
df_prot_phe['display_relation'] = 'associated with'
df_prot_phe = clean_edges(df_prot_phe)

### Effect effect interactions (HPO)

In [ ]:
df_phe_phe = pd.merge(df_hp_parents, df_hp_terms, 'left', left_on='parent', right_on='id')
df_phe_phe = df_phe_phe.rename(columns={'name':'parent_name'})
df_phe_phe = pd.merge(df_phe_phe, df_hp_terms, 'left', left_on='child', right_on='id')
df_phe_phe = df_phe_phe.rename(columns={'name':'child_name'})
df_phe_phe = df_phe_phe.get(['parent', 'child', 'parent_name', 'child_name'])

df_phe_phe = df_phe_phe.rename(columns={'parent':'x_id', 'child':'y_id', 'parent_name':'x_name', 'child_name':'y_name'})
df_phe_phe['x_type'] = 'effect/phenotype'
df_phe_phe['x_source'] = 'HPO'
df_phe_phe['y_type'] = 'effect/phenotype'
df_phe_phe['y_source'] = 'HPO'
df_phe_phe['relation'] = 'phenotype_phenotype'
df_phe_phe['display_relation'] = 'parent-child'
df_phe_phe = clean_edges(df_phe_phe)

### Disease effect interactions (HPO-A)

In [ ]:
df_dis_phe_pos1 = pd.merge(df_hpoa_pos, df_mondo_xref, "left", left_on="disease_ontology_id", right_on="ontology_id")
df_dis_phe_pos1 = df_dis_phe_pos1.query(
    '(disease_ontology == ontology) or (disease_ontology == "ORPHA" and ontology == "Orphanet")'
)
df_dis_phe_pos1 = pd.merge(df_dis_phe_pos1, df_hp_terms, "left", left_on="hp_id", right_on="id").rename(
    columns={"name": "hp_name"}
)
df_dis_phe_pos1 = pd.merge(df_dis_phe_pos1, df_mondo_terms, "left", left_on="mondo_id", right_on="id").rename(
    columns={"name": "mondo_name"}
)
df_dis_phe_pos1 = df_dis_phe_pos1[["mondo_id", "mondo_name", "hp_id", "hp_name"]]
df_dis_phe_pos1 = df_dis_phe_pos1.rename(
    columns={"mondo_id": "x_id", "mondo_name": "x_name", "hp_id": "y_id", "hp_name": "y_name"}
)
df_dis_phe_pos1 = df_dis_phe_pos1.assign(
    x_source="MONDO",
    x_type="disease",
    y_source="HPO",
    y_type="effect/phenotype",
    relation="disease_phenotype_positive",
    display_relation="phenotype present",
)
df_dis_phe_pos1 = clean_edges(df_dis_phe_pos1)


In [ ]:
df_dis_phe_neg = pd.merge(df_hpoa_neg, df_mondo_xref, "left", left_on="disease_ontology_id", right_on="ontology_id")
df_dis_phe_neg = df_dis_phe_neg.query(
    '(disease_ontology == ontology) or (disease_ontology == "ORPHA" and ontology == "Orphanet")'
)
df_dis_phe_neg = pd.merge(df_dis_phe_neg, df_hp_terms, "left", left_on="hp_id", right_on="id").rename(
    columns={"name": "hp_name"}
)
df_dis_phe_neg = pd.merge(df_dis_phe_neg, df_mondo_terms, "left", left_on="mondo_id", right_on="id").rename(
    columns={"name": "mondo_name"}
)
df_dis_phe_neg = df_dis_phe_neg[["mondo_id", "mondo_name", "hp_id", "hp_name"]]
df_dis_phe_neg = df_dis_phe_neg.rename(
    columns={"mondo_id": "x_id", "mondo_name": "x_name", "hp_id": "y_id", "hp_name": "y_name"}
)
df_dis_phe_neg = df_dis_phe_neg.assign(
    x_source="MONDO",
    x_type="disease",
    y_source="HPO",
    y_type="effect/phenotype",
    relation="disease_phenotype_negative",
    display_relation="phenotype absent",
)
df_dis_phe_neg = clean_edges(df_dis_phe_neg)


### Remove phenotype nodes if they exist in MONDO

In [ ]:
# phenotypes that are actually diseases in MONDO
# avoid duplicate nodes and convert them to disease relations

# Use curated df_mondo_xref_hp (MONDO:9813/18767 removed above), not raw df_mondo_xref HP rows.
mondo_xref_hp_subset = df_mondo_xref_hp.copy()
mondo_xref_hp_subset.loc[:, 'ontology_id'] = mondo_xref_hp_subset.get('ontology_id').astype(int).astype(str).values
assert mondo_xref_hp_subset.ontology_id.duplicated().sum() == 0

In [ ]:
# "inner" is the key for the need to use merge command here
hp_ids_r_mondo = pd.merge(mondo_xref_hp_subset, df_hp_terms, 'inner', left_on='ontology_id', right_on='id').get('ontology_id').values
hp_ids_r_mondo[:5]

In [ ]:
def replace_hp_data_w_mondo(df, hp_id_col, drop_cols=[], mondo_xref_hp_subset=mondo_xref_hp_subset, df_mondo_terms=df_mondo_terms): 
    cols = list(df.columns.values)
    cols.extend(['mondo_id', 'mondo_name'])
    [cols.remove(x) for x in drop_cols]
    df = pd.merge(df, mondo_xref_hp_subset, 'left', left_on=hp_id_col, right_on='ontology_id')
    df = pd.merge(df, df_mondo_terms, 'left', left_on='mondo_id', right_on='id')
    df = df.rename(columns={'name':'mondo_name'}).get(cols)
    return df

In [ ]:
pd.set_option("display.max_columns", None)
pd.reset_option("display.max_columns")

In [ ]:
# HANDLE EFFECT EFFECT 
# PHE-PHE should be PHE-DIS if ONE PHE is in MONDO
#updated
df_dis_phe_x = df_phe_phe.query('x_id in @hp_ids_r_mondo and y_id not in @hp_ids_r_mondo')
df_dis_phe_x = replace_hp_data_w_mondo(df=df_dis_phe_x, hp_id_col='x_id', 
                                       drop_cols=[c for c in df_dis_phe_x.columns.values if 'x_' in c])
df_dis_phe_x = df_dis_phe_x.rename(columns={'mondo_id':'x_id', 'mondo_name':'x_name'})
df_dis_phe_x['x_source'] = 'MONDO'
df_dis_phe_x['x_type'] = 'disease'

df_dis_phe_y = df_phe_phe.query('y_id in @hp_ids_r_mondo and x_id not in @hp_ids_r_mondo')
df_dis_phe_y = replace_hp_data_w_mondo(df=df_dis_phe_y, hp_id_col='y_id',
                                       drop_cols=[c for c in df_dis_phe_y.columns.values if 'y_' in c])
df_dis_phe_y = df_dis_phe_y.rename(columns={'mondo_id':'y_id', 'mondo_name':'y_name'})
df_dis_phe_y['y_source'] = 'MONDO'
df_dis_phe_y['y_type'] = 'disease'

df_dis_phe_pos2 = pd.concat([df_dis_phe_x, df_dis_phe_y], ignore_index=True)
df_dis_phe_pos2['relation'] = 'disease_phenotype_positive'
df_dis_phe_pos2.loc[:, 'display_relation'] = 'phenotype present'
df_dis_phe_pos2 = clean_edges(df_dis_phe_pos2)

In [ ]:
# PHE-PHE should be DIS-DIS if BOTH PHE are in MONDO
#updated
df_dis_dis2 = df_phe_phe.query('x_id in @hp_ids_r_mondo and y_id in @hp_ids_r_mondo')
df_dis_dis2 = replace_hp_data_w_mondo(df=df_dis_dis2, 
                                       hp_id_col='x_id', 
                                       drop_cols=[c for c in df_dis_dis2.columns.values if 'x_' in c])
df_dis_dis2 = df_dis_dis2.rename(columns={'mondo_id':'x_id', 'mondo_name':'x_name'})
df_dis_dis2 = replace_hp_data_w_mondo(df=df_dis_dis2, 
                                       hp_id_col='y_id', 
                                       drop_cols=[c for c in df_dis_dis2.columns.values if 'y_' in c])
df_dis_dis2 = df_dis_dis2.rename(columns={'mondo_id':'y_id', 'mondo_name':'y_name'})
df_dis_dis2['x_source'] = 'MONDO'
df_dis_dis2['x_type'] = 'disease'
df_dis_dis2['y_source'] = 'MONDO'
df_dis_dis2['y_type'] = 'disease'
df_dis_dis2['relation'] = 'disease_disease'
df_dis_dis2['display_relation'] = 'parent-child'
df_dis_dis2 = clean_edges(df_dis_dis2)

# drop relations in PHE PHE if either PHE is in MONDO
# phenotype phenotype should have no disease nodes
df_phe_phe = df_phe_phe.query('x_id not in @hp_ids_r_mondo and y_id not in @hp_ids_r_mondo')

In [ ]:
# HANDLE PROTEIN EFFECT 
# if phenotype in MONDO make it protein-disease relations 
#updated
df_prot_dis2= df_prot_phe.query('y_id in @hp_ids_r_mondo')
df_prot_dis2 = replace_hp_data_w_mondo(df=df_prot_dis2, hp_id_col='y_id',
                                       drop_cols=[c for c in df_prot_dis2.columns.values if 'y_' in c])
df_prot_dis2 = df_prot_dis2.rename(columns={'mondo_id':'y_id', 'mondo_name':'y_name'})
df_prot_dis2['y_source'] = 'MONDO'
df_prot_dis2['y_type'] = 'disease'
df_prot_dis2['relation'] = 'disease_protein'
df_prot_dis2['display_relation'] = 'associated with'
df_prot_dis2 = clean_edges(df_prot_dis2)

# remove from protein-phenotype if phenotype in MONDO 
df_prot_phe = df_prot_phe.query('y_id not in @hp_ids_r_mondo')

In [ ]:
# HANDLE DISEASE EFFECT 
#updated
# remove from protein-phenotype if phenotype in MONDO 
df_dis_phe_pos1 = df_dis_phe_pos1.query('y_id not in @hp_ids_r_mondo')

# NEGATIVE disease_phenotype should just be dropped because negative disease_disease doesn't make sense 
df_dis_phe_neg = df_dis_phe_neg.query('y_id not in @hp_ids_r_mondo')

In [ ]:
# COMBINE DATAFRAMES 
#updated
df_prot_dis = pd.concat([df_prot_dis1, df_prot_dis2], ignore_index=True).drop_duplicates()

In [ ]:
df_dis_dis = pd.concat([df_dis_dis1, df_dis_dis2], ignore_index=True).drop_duplicates()
df_dis_phe_pos = pd.concat([df_dis_phe_pos1, df_dis_phe_pos2], ignore_index=True).drop_duplicates()

### Drug effect interactions (SIDER)

In [ ]:
#please note that df_sider does not get updated
df_drug_effect = pd.merge(df_sider, df_db_atc, 'left', left_on='atc', right_on='atc_code')
df_drug_effect = df_drug_effect.rename(columns={'parent_key':'DrugBank', 'UMLS_from_meddra':'UMLS'})
df_drug_effect = pd.merge(df_drug_effect, db_vocab, 'left', left_on='DrugBank', right_on='DrugBank ID')
df_drug_effect = pd.merge(df_drug_effect, df_hp_xref, 'left', left_on='UMLS' , right_on='ontology_id')
df_drug_effect = pd.merge(df_drug_effect, df_hp_terms, 'left', left_on='hp_id' , right_on='id')
df_drug_effect = df_drug_effect.get(['DrugBank ID','Common name','hp_id', 'name'])
df_drug_effect = df_drug_effect.dropna().drop_duplicates()

df_drug_effect = df_drug_effect.rename(columns={'DrugBank ID':'x_id', 'Common name':'x_name', 'hp_id':'y_id', 'name':'y_name'})
df_drug_effect['x_type'] = 'drug'
df_drug_effect['x_source'] = 'DrugBank'
df_drug_effect['y_type'] = 'effect/phenotype'
df_drug_effect['y_source'] = 'HPO'
df_drug_effect['relation'] = 'drug_effect'
df_drug_effect['display_relation'] = 'side effect'
df_drug_effect = df_drug_effect.query('y_id not in @hp_ids_r_mondo')
df_drug_effect = clean_edges(df_drug_effect)


## GO Terms

### Go terms interactions (GO)

In [ ]:
bp = df_go_terms.query('go_term_type=="biological_process"')
df_bp_bp = pd.merge(df_go_edges, bp, 'inner', left_on='x', right_on='go_term_id')
df_bp_bp = df_bp_bp.rename(columns={'go_term_id':'x_id','go_term_name':'x_name','go_term_type':'x_type'})
df_bp_bp = pd.merge(df_bp_bp, bp, 'inner', left_on='y', right_on='go_term_id')
df_bp_bp = df_bp_bp.rename(columns={'go_term_id':'y_id','go_term_name':'y_name','go_term_type':'y_type'})
df_bp_bp['relation'] = 'bioprocess_bioprocess'
df_bp_bp['x_source'] = 'GO'
df_bp_bp['y_source'] = 'GO'
df_bp_bp['display_relation'] = 'parent-child'
df_bp_bp = clean_edges(df_bp_bp)

In [ ]:
mf = df_go_terms.query('go_term_type=="molecular_function"')
df_mf_mf = pd.merge(df_go_edges, mf, 'inner', left_on='x', right_on='go_term_id')
df_mf_mf = df_mf_mf.rename(columns={'go_term_id':'x_id','go_term_name':'x_name','go_term_type':'x_type'})
df_mf_mf = pd.merge(df_mf_mf, mf, 'inner', left_on='y', right_on='go_term_id')
df_mf_mf = df_mf_mf.rename(columns={'go_term_id':'y_id','go_term_name':'y_name','go_term_type':'y_type'})
df_mf_mf['relation'] = 'molfunc_molfunc'
df_mf_mf['display_relation'] = 'parent-child'
df_mf_mf['x_source'] = 'GO'
df_mf_mf['y_source'] = 'GO'
df_mf_mf = clean_edges(df_mf_mf)

In [ ]:
cc = df_go_terms.query('go_term_type=="cellular_component"')
df_cc_cc = pd.merge(df_go_edges, cc, 'inner', left_on='x', right_on='go_term_id')
df_cc_cc = df_cc_cc.rename(columns={'go_term_id':'x_id','go_term_name':'x_name','go_term_type':'x_type'})
df_cc_cc = pd.merge(df_cc_cc, cc, 'inner', left_on='y', right_on='go_term_id')
df_cc_cc = df_cc_cc.rename(columns={'go_term_id':'y_id','go_term_name':'y_name','go_term_type':'y_type'})
df_cc_cc['relation'] = 'cellcomp_cellcomp'
df_cc_cc['display_relation'] = 'parent-child'
df_cc_cc['x_source'] = 'GO'
df_cc_cc['y_source'] = 'GO'
df_cc_cc = clean_edges(df_cc_cc)

### Go protein interactions (Gene2GO)

In [ ]:
#updated
df_prot_path = pd.merge(df_gene2go, df_go_terms, 'inner', 'go_term_id').rename(columns={'go_term_type_x':'go_term_type'})
df_prot_path = pd.merge(df_prot_path, df_prot_names, 'left', left_on='ncbi_gene_id', right_on='ncbi_id')
df_prot_path = df_prot_path.rename(columns={'ncbi_gene_id':'x_id', 'symbol':'x_name', 
                             'go_term_id':'y_id','go_term_name':'y_name', 'go_term_type':'y_type'})
df_prot_path['x_type'] = 'gene/protein'
df_prot_path['x_source'] = 'NCBI'
df_prot_path['y_source'] = 'GO'
df_prot_path = df_prot_path.get(['x_id','x_type', 'x_name', 'x_source','y_id','y_type', 'y_name', 'y_source'])

df_prot_mf = df_prot_path.query('y_type=="molecular_function"').copy()
df_prot_mf['relation'] = 'molfunc_protein'
df_prot_mf['display_relation'] = 'interacts with'
df_prot_mf = clean_edges(df_prot_mf)
df_prot_cc = df_prot_path.query('y_type=="cellular_component"').copy()
df_prot_cc['relation'] = 'cellcomp_protein'
df_prot_cc['display_relation'] = 'interacts with'
df_prot_cc = clean_edges(df_prot_cc)
df_prot_bp = df_prot_path.query('y_type=="biological_process"').copy()
df_prot_bp['relation'] = 'bioprocess_protein'
df_prot_bp['display_relation'] = 'interacts with'
df_prot_bp = clean_edges(df_prot_bp)

## Exposure

### Exposure protein interactions (CTD)

In [ ]:
#updated 
df_exp_prot = df_exposures.get(['exposurestressorname', 'exposurestressorid','exposuremarker', 'exposuremarkerid'])
df_exp_prot = df_exp_prot.loc[df_exp_prot.get(['exposuremarkerid']).dropna().index, :]

gene_row_index = []
for idx, data in df_exp_prot.iterrows():
    if data.exposuremarkerid.isnumeric(): 
        gene_row_index.append(idx)

df_exp_prot = df_exp_prot.loc[gene_row_index, :].astype({'exposuremarkerid': 'int'}).astype({'exposuremarkerid': 'str'})
df_exp_prot = pd.merge(df_exp_prot, df_prot_names, 'left', left_on='exposuremarkerid', right_on='ncbi_id')

df_exp_prot = df_exp_prot.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 'ncbi_id':'y_id', 'symbol':'y_name'})
df_exp_prot['x_type'] = 'exposure'
df_exp_prot['x_source'] = 'CTD'
df_exp_prot['y_type'] = 'gene/protein'
df_exp_prot['y_source'] = 'NCBI'
df_exp_prot['relation'] = 'exposure_protein'
df_exp_prot['display_relation'] = 'interacts with'
df_exp_prot = clean_edges(df_exp_prot)

### Exposure disease interactions (CTD)

In [ ]:
#updated
df_exp_dis = df_exposures.get(['exposurestressorname', 'exposurestressorid','diseasename', 'diseaseid'])
df_exp_dis = df_exp_dis.loc[df_exp_dis.get(['diseaseid']).dropna().index, :]
df_exp_dis = pd.merge(df_exp_dis, df_mondo_xref.query('ontology=="MESH"'), 'left', left_on='diseaseid', right_on='ontology_id')
df_exp_dis = pd.merge(df_exp_dis, df_mondo_terms, 'left', left_on='mondo_id', right_on= 'id')

df_exp_dis = df_exp_dis.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 'mondo_id':'y_id', 'name':'y_name'})
df_exp_dis['x_type'] = 'exposure'
df_exp_dis['x_source'] = 'CTD'
df_exp_dis['y_type'] = 'disease'
df_exp_dis['y_source'] = 'MONDO'
df_exp_dis['relation'] = 'exposure_disease'
df_exp_dis['display_relation'] = 'linked to'
df_exp_dis = clean_edges(df_exp_dis)
df_exp_dis.head(1), len(df_exp_dis)

### Exposure exposure interactions (CTD)

In [ ]:
#updated
exposures = np.unique(df_exposures.get('exposurestressorid').values)
df_exp_exp = df_exposures.query('exposuremarkerid in @exposures')

df_exp_exp = df_exp_exp.get(['exposurestressorname', 'exposurestressorid','exposuremarker', 'exposuremarkerid'])
df_exp_exp = df_exp_exp.loc[df_exp_exp.get(['exposuremarkerid']).dropna().index, :]
df_exp_exp = df_exp_exp.drop_duplicates()

df_exp_exp = df_exp_exp.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 'exposuremarker':'y_name', 'exposuremarkerid':'y_id'})
df_exp_exp['x_type'] = 'exposure'
df_exp_exp['x_source'] = 'CTD'
df_exp_exp['y_type'] = 'exposure'
df_exp_exp['y_source'] = 'CTD'
df_exp_exp['relation'] = 'exposure_exposure'
df_exp_exp['display_relation'] = 'parent-child'
df_exp_exp = clean_edges(df_exp_exp)

### Exposure pathway interactions (CTD)

In [ ]:
# phenotypes are actually pathways 
#updated
df_exp_path = df_exposures.get(['exposurestressorname', 'exposurestressorid','phenotypename', 'phenotypeid'])
df_exp_path = df_exp_path.loc[df_exp_path.get(['phenotypeid']).dropna().index, :]
df_exp_path.loc[:, 'phenotypeid'] = [str(int(x.split(':')[1])) for x in df_exp_path.get(['phenotypeid']).values.reshape(-1)]
df_exp_path = df_exp_path.drop_duplicates()
df_exp_path = pd.merge(df_exp_path, df_go_terms, 'inner', left_on='phenotypeid', right_on='go_term_id')
df_exp_path = df_exp_path.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 
                                          'go_term_id':'y_id', 'go_term_name':'y_name', 'go_term_type':'y_type'})
df_exp_path['x_type'] = 'exposure'
df_exp_path['x_source'] = 'CTD'
df_exp_path['y_source'] = 'GO'
df_exp_bp = df_exp_path.query('y_type=="biological_process"').copy()
df_exp_bp['relation'] = 'exposure_bioprocess'
df_exp_bp['display_relation'] = 'interacts with'
df_exp_bp = clean_edges(df_exp_bp)

In [ ]:
df_exp_mf = df_exp_path.query('y_type=="molecular_function"').copy()
df_exp_mf['relation'] = 'exposure_molfunc'
df_exp_mf['display_relation'] = 'interacts with'
df_exp_mf = clean_edges(df_exp_mf)

In [ ]:
df_exp_cc = df_exp_path.query('y_type=="cellular_component"').copy()
df_exp_cc['relation'] = 'exposure_cellcomp'
df_exp_cc['display_relation'] = 'interacts with'
df_exp_cc = clean_edges(df_exp_cc)

## Anatomy

### Anatomy anatomy interactions (UBERON) 

In [ ]:
#updated
df_ana_ana = pd.merge(df_uberon_is_a, df_uberon_terms, 'left', left_on='id', right_on='id')
df_ana_ana = df_ana_ana.rename(columns={'id':'x_id', 'name':'x_name'})
df_ana_ana = pd.merge(df_ana_ana, df_uberon_terms, 'left', left_on='is_a', right_on='id')
df_ana_ana = df_ana_ana.rename(columns={'id':'y_id', 'name':'y_name'})
df_ana_ana['x_type'] = 'anatomy'
df_ana_ana['x_source'] = 'UBERON'
df_ana_ana['y_type'] = 'anatomy'
df_ana_ana['y_source'] = 'UBERON'
df_ana_ana['relation'] = 'anatomy_anatomy'
df_ana_ana['display_relation'] = 'parent-child'
df_ana_ana = clean_edges(df_ana_ana)

### Anatomy Protein (BGEE)

In [ ]:
#MUST get a fresh value of df_bgee
df_bgee = pd.read_csv(data_path+'bgee/anatomy_gene.csv', low_memory=False)
df_bgee = df_bgee.astype({'expression_rank':int}).astype({'expression_rank':str})
df_bgee = df_bgee.astype({'anatomy_id':int}).astype({'anatomy_id':str})
df_bgee = df_bgee[pd.to_numeric(df_bgee["expression_rank"], errors="coerce") <= expression_rank]
len(df_bgee)
assert_dtypes(df_bgee)

#updated
updated_df_bgee = pd.merge(df_bgee, df_prot_names, 'inner', left_on='gene_name', right_on='symbol')
updated_df_bgee = updated_df_bgee.rename(columns={'ncbi_id':'x_id', 'symbol':'x_name', 
                                  'anatomy_id':'y_id', 'anatomy_name':'y_name'})
updated_df_bgee['x_source'] = 'NCBI'
updated_df_bgee['x_type'] = 'gene/protein'
updated_df_bgee['y_source'] = 'UBERON'
updated_df_bgee['y_type'] = 'anatomy'
df_ana_prot_pos = updated_df_bgee[updated_df_bgee.expression=="present"].copy()
df_ana_prot_pos['relation'] = 'anatomy_protein_present'
df_ana_prot_pos['display_relation'] = 'expression present'
df_ana_prot_pos = clean_edges(df_ana_prot_pos)
df_ana_prot_neg = updated_df_bgee[updated_df_bgee.expression=="absent"].copy()
df_ana_prot_neg['relation'] = 'anatomy_protein_absent'
df_ana_prot_neg['display_relation'] = 'expression absent'
df_ana_prot_neg = clean_edges(df_ana_prot_neg)

## Pathways

In [ ]:
#updated
df_path_path = pd.merge(df_reactome_rels, df_reactome_terms, 'inner', left_on='reactome_id_1', right_on='reactome_id')
df_path_path = df_path_path.rename(columns={'reactome_id': 'x_id', 'reactome_name':'x_name'})
df_path_path = pd.merge(df_path_path, df_reactome_terms, 'inner', left_on='reactome_id_2', right_on='reactome_id')
df_path_path = df_path_path.rename(columns={'reactome_id': 'y_id', 'reactome_name':'y_name'})

df_path_path['x_source'] = 'REACTOME'
df_path_path['x_type'] = 'pathway'
df_path_path['y_source'] = 'REACTOME'
df_path_path['y_type'] = 'pathway'
df_path_path['relation'] = 'pathway_pathway'
df_path_path['display_relation'] = 'parent-child'
df_path_path = clean_edges(df_path_path)

### Pathway protein interactions

In [ ]:
#updated
df_path_prot = pd.merge(df_reactome_ncbi, df_prot_names, 'inner', 'ncbi_id')

df_path_prot = df_path_prot.rename(columns={'ncbi_id': 'x_id', 'symbol':'x_name', 
                                            'reactome_id': 'y_id', 'reactome_name':'y_name'})
df_path_prot['x_source'] = 'NCBI'
df_path_prot['x_type'] = 'gene/protein'
df_path_prot['y_source'] = 'REACTOME'
df_path_prot['y_type'] = 'pathway'
df_path_prot['relation'] = 'pathway_protein'
df_path_prot['display_relation'] = 'interacts with'
df_path_prot = clean_edges(df_path_prot)

## Phase 3 — Compile the unified knowledge graph (`kg`)

- Concatenate all standardized edge tables (`df_*`) built in **Phase 2**.
- Run overlap / sanity checks (e.g. `check_name_overlap`) to catch stray ontology “junk” labels in node names.
- `drop_duplicates`, drop nulls, and remove self-loops to produce the final **`kg`** export.


In [ ]:
def check_name_overlap(df, check_list, name_col='name', return_overlap=False):
    if name_col not in df.columns:
        raise ValueError(f"Column '{name_col}' not found in dataframe. Available columns: {list(df.columns)}")
    
    unique_names = set(df[name_col].dropna().unique())
    
    check_set = set(check_list)
    
    overlap = unique_names.intersection(check_set)
    
    if return_overlap:
        return overlap
    else:
        return len(overlap) > 0, len(overlap)

In [ ]:
# Compile all edge dataframes into a unified knowledge graph
# Concatenates all 28 edge dataframes from different data sources into a single knowledge graph.
# Then creates reverse edges (bidirectional) for all relations to enable traversal in both directions.
# Removes duplicates, null values, and self-loops to ensure graph quality.

kg = pd.concat([df_prot_prot, df_prot_drug, df_drug_dis, df_drug_drug, df_prot_phe,
                df_phe_phe, df_dis_phe_neg, df_dis_phe_pos, df_prot_dis, df_dis_dis, 
                df_drug_effect, df_bp_bp, df_mf_mf, df_cc_cc, df_prot_mf, 
                df_prot_cc, df_prot_bp, df_exp_prot, df_exp_dis, df_exp_exp, 
                df_exp_bp, df_exp_mf, df_exp_cc, df_path_path, df_path_prot,
                df_ana_ana, df_ana_prot_pos, df_ana_prot_neg]) #28
kg = kg.drop_duplicates()
kg_no_rev = kg.copy()
kg_rev = kg.copy().rename(columns={'x_id':'y_id','x_type':'y_type', 'x_name':'y_name', 'x_source':'y_source',
                            'y_id':'x_id','y_type':'x_type', 'y_name':'x_name', 'y_source':'x_source' }) #add reverse edges
early_kg_csv = os.path.normpath(os.path.join(data_path, '..', f'{RUN_DATE}-primekg_plus.csv'))
kg.to_csv(early_kg_csv, index=False)
kg = pd.concat([kg, kg_rev])

kg = kg.drop_duplicates()

kg = kg.dropna()
# remove self loops from edges 
kg = kg.query('not ((x_id == y_id) and (x_type == y_type) and (x_source == y_source) and (x_name == y_name))')

updated_total = len(kg)

## Giant connected component

Round-trip `kg` through `kg_raw.csv`, index nodes and edges, extract the largest connected component, and save `auxillary/{RUN_DATE}-kg_giant.csv` for disease grouping in notebook **02**.

In [ ]:
import os
_aux = os.path.join(save_path, "auxillary")
os.makedirs(_aux, exist_ok=True)
# # Match `build_graph_original.ipynb`: round-trip `kg` through `kg_raw.csv` before giant-component indexing.
kg.to_csv(os.path.join(_aux, f"{RUN_DATE}-kg_raw.csv"), index=False)
kg = pd.read_csv(os.path.join(_aux, f"{RUN_DATE}-kg_raw.csv"), low_memory=False) # raw version has no x_index and y_index

In [ ]:
nodes = pd.concat([kg.get(['x_id','x_type', 'x_name','x_source']).rename(columns={'x_id':'node_id', 'x_type':'node_type', 'x_name':'node_name','x_source':'node_source'}), 
                   kg.get(['y_id','y_type', 'y_name','y_source']).rename(columns={'y_id':'node_id', 'y_type':'node_type', 'y_name':'node_name','y_source':'node_source'})])

nodes = nodes.drop_duplicates().reset_index().drop('index',axis=1).reset_index().rename(columns={'index':'node_idx'})

In [ ]:
edges = pd.merge(kg, nodes, 'left', left_on=['x_id','x_type', 'x_name','x_source'], right_on=['node_id','node_type','node_name','node_source'])
edges = edges.rename(columns={'node_idx':'x_idx'})
edges = pd.merge(edges, nodes, 'left', left_on=['y_id','y_type', 'y_name','y_source'], right_on=['node_id','node_type','node_name','node_source'])
edges = edges.rename(columns={'node_idx':'y_idx'})
edges = edges.get(['relation', 'display_relation','x_idx', 'y_idx'])
edges['combine_idx'] = edges['x_idx'].astype(str) + '-' + edges['y_idx'].astype(str)
edge_index = edges.get(['x_idx', 'y_idx']).values.T

In [ ]:
graph = ig.Graph()
graph.add_vertices(list(range(nodes.shape[0])))
graph.add_edges([tuple(x) for x in edge_index.T])

graph = graph.as_undirected(mode='collapse')

c = graph.components(mode='strong')
giant = c.giant()


assert not giant.is_directed()
assert giant.is_connected()

giant_nodes = giant.vs['name']
new_nodes = nodes.query('node_idx in @giant_nodes')
assert new_nodes.shape[0] == giant.vcount()

new_edges = edges.query('x_idx in @giant_nodes and y_idx in @giant_nodes').copy()
assert new_edges.shape[0] == giant.ecount()

new_kg = pd.merge(new_edges, new_nodes, 'left', left_on='x_idx', right_on='node_idx')
new_kg = new_kg.rename(columns={'node_id':'x_id', 'node_type':'x_type', 'node_name':'x_name','node_source':'x_source'}) 
new_kg = pd.merge(new_kg, new_nodes, 'left', left_on='y_idx', right_on='node_idx')
new_kg = new_kg.rename(columns={'node_id':'y_id', 'node_type':'y_type', 'node_name':'y_name','node_source':'y_source'}) 
new_kg = clean_edges(new_kg)

In [ ]:
kg = new_kg.copy()
kg.to_csv(save_path + f'auxillary/{RUN_DATE}-kg_giant.csv', index=False)

## Outputs

- `datasets/data/kg/auxillary/{RUN_DATE}-kg_raw.csv`
- `datasets/data/kg/auxillary/{RUN_DATE}-kg_giant.csv`

Continue with **`02_disease_grouping.ipynb`**.
